# 00 | Evidence, scope and the case logic

**Author: Chanakya**

Start here. Audit the locked inputs, establish units and map the brief before fitting models. A source download is not a verified operating metric.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='00_evidence_and_case'
shared.ACTIVE_SOURCES=[]

Offline inputs: raw-v5-2026-09-14 | Author: Chanakya


## 1. Verify the frozen research inputs
The checksum verifier is read-only. It also checks provenance records locked at collection close.

In [2]:
import subprocess
v = subprocess.run([sys.executable, str(ROOT/'scripts/verify_release.py')], capture_output=True, text=True, check=True)
print(v.stdout)
inv = pd.DataFrame(json.loads((ROOT/'data/manifests/raw_inventory.json').read_text()))
quality = pd.DataFrame(json.loads((ROOT/'data/manifests/source_validation.json').read_text()))
coverage = inv.groupby('format').agg(files=('file','size'),MB=('bytes',lambda x:x.sum()/1e6)).reset_index()
display(table(coverage,'00_raw_coverage'))
plt.figure(figsize=(9,4));plt.barh(coverage['format'],coverage['files']);plt.xlabel('Preserved files');plt.title('Evidence volume is not evidence strength');fig('00_evidence_coverage','Source: locked raw-v5 inventory. Includes failed-response bodies.')

{
  "release_id": "raw-v5-2026-09-14",
  "status": "PASS",
  "raw_files_checked": 397,
  "errors": []
}



,format,files,MB
0,Browser table TSV,1,0.006
1,CSV,8,2.240
2,Google Play public RPC,75,12.126
3,HTML,147,37.078
4,JSON,41,0.978
5,PDF,92,252.270
6,Text or JavaScript,33,0.589


<Figure size 900x400 with 1 Axes>

## 2. Brief inputs and denominator controls
Page numbers below are physical PDF pages. The illustrative audience bars are deliberately excluded from market sizing. The repurchase horizon and renewal cohort are undefined. ₹89 is a midpoint tournament-price scenario, not an observed single-match price.

In [3]:
brief = PdfReader(ROOT/'data/raw/case_inputs/case_brief/BGCC_FANCODE_R3.pdf')
inputs=[('registered_users',240e6,'users, cumulative',9,'Not active or reachable audience'),('watch_minutes',90,'minutes per sport/tour',9,'No payer or session denominator'),('masters_uplift',.5,'relative uplift',9,'Association, not causal timing effect'),('noncore_share',.1,'ordinary-week tennis viewers',9,'Not platform conversion'),('repeat_low',.60,'repurchase proportion',10,'Horizon unknown'),('repeat_high',.65,'repurchase proportion',10,'Not a stationary repeat hazard'),('renewal',.9,'season renewal proportion',10,'Cohort and period unspecified'),('cac_low',150,'INR per paying subscriber target',11,'Not observed incremental CAC'),('cac_high',200,'INR per paying subscriber target',11,'Target may exceed product contribution'),('contest_cvr',.078,'click-to-purchase proportion',11,'Historical case channel only'),('tournament_low',79,'INR per tournament',11,'Case input, current checkout unverified'),('tournament_high',99,'INR per tournament',11,'Case input'),('season_price',399,'INR per season',11,'Declines late season, not rolling 365 days'),('performance_share_low',.65,'fraction of total marketing spend',11,'Current case baseline, not proposed allocation'),('performance_share_high',.70,'fraction of total marketing spend',11,'Current case baseline, not proposed allocation'),('attention_lead_days',2,'days before match',11,'Approximate engagement peak, not measured purchase-conversion peak'),('weekly_inventory',200,'tour-level matches or sessions per week',9,'Case input, not independently counted')]
b = pd.DataFrame(inputs,columns=['input_id','value','unit','pdf_page','limitation']);b['source_id']='supplied_BGCC_FANCODE_R3'
table(b,'case_inputs',True);display(table(b,'00_case_dictionary'))
operating=pd.DataFrame([
('performance_marketing','65–70% of total marketing spend','Current baseline'),
('brand_other','30–35% of total marketing spend, chart illustrates 68/32','Complementary current baseline'),
('campaign_attention','Engagement peaks roughly two days before the match','Plan occasion-led activation around D-2, do not claim a purchase peak'),
('contest_conversion','7.8% contest link-click to pass purchase, recurring','Use supplied rate for this channel, not every acquisition channel'),
('publisher_takeovers','ESPN.in inventory takeovers already used','Improve the existing channel rather than propose it as new'),
('native_personalities','Effectively zero incremental media-rights cost','Do not add an ATP rights surcharge. Production, talent and distribution costs are separate'),
('portfolio_growth','Incremental subscription gains skew toward F1 and football','Accepted case fact, no external corroboration required for its use'),
('season_price','Typically INR399, declining toward season end','Use remaining-season coverage, not rolling annual validity')],columns=['case_fact','supplied_value','decision_use']);operating['source_id']='supplied_BGCC_FANCODE_R3';display(table(operating,'00_case_operating_facts'))
assert '7.8%' in brief.pages[10].extract_text()
assert '65-70%' in brief.pages[10].extract_text()
assert '2 days' in brief.pages[10].extract_text()
comparison=pd.DataFrame({'reading':['Stated uplift','Illustrative bars'],'ordinary_index':[100,100],'spike_index':[150,500]})
comparison['uplift_pct']=(comparison.spike_index/comparison.ordinary_index-1)*100
display(table(comparison,'00_spike_definition_audit'))
print('The bar uplift is 8 times the stated uplift, but the bars are explicitly illustrative. Neither is a measured acquisition multiplier.')

,input_id,value,unit,pdf_page,limitation,source_id
0,registered_users,"240,000,000.000","users, cumulative",9,Not active or reachable audience,supplied_BGCC_FANCODE_R3
1,watch_minutes,90.000,minutes per sport/tour,9,No payer or session denominator,supplied_BGCC_FANCODE_R3
2,masters_uplift,0.500,relative uplift,9,"Association, not causal timing effect",supplied_BGCC_FANCODE_R3
3,noncore_share,0.100,ordinary-week tennis viewers,9,Not platform conversion,supplied_BGCC_FANCODE_R3
4,repeat_low,0.600,repurchase proportion,10,Horizon unknown,supplied_BGCC_FANCODE_R3
5,repeat_high,0.650,repurchase proportion,10,Not a stationary repeat hazard,supplied_BGCC_FANCODE_R3
6,renewal,0.900,season renewal proportion,10,Cohort and period unspecified,supplied_BGCC_FANCODE_R3
7,cac_low,150.000,INR per paying subscriber target,11,Not observed incremental CAC,supplied_BGCC_FANCODE_R3
8,cac_high,200.000,INR per paying subscriber target,11,Target may exceed product contribution,supplied_BGCC_FANCODE_R3
9,contest_cvr,0.078,click-to-purchase proportion,11,Historical case channel only,supplied_BGCC_FANCODE_R3


,case_fact,supplied_value,decision_use,source_id
0,performance_marketing,65–70% of total marketing spend,Current baseline,supplied_BGCC_FANCODE_R3
1,brand_other,"30–35% of total marketing spend, chart illustr...",Complementary current baseline,supplied_BGCC_FANCODE_R3
2,campaign_attention,Engagement peaks roughly two days before the m...,"Plan occasion-led activation around D-2, do no...",supplied_BGCC_FANCODE_R3
3,contest_conversion,"7.8% contest link-click to pass purchase, recu...","Use supplied rate for this channel, not every ...",supplied_BGCC_FANCODE_R3
4,publisher_takeovers,ESPN.in inventory takeovers already used,Improve the existing channel rather than propo...,supplied_BGCC_FANCODE_R3
5,native_personalities,Effectively zero incremental media-rights cost,Do not add an ATP rights surcharge. Production...,supplied_BGCC_FANCODE_R3
6,portfolio_growth,Incremental subscription gains skew toward F1 ...,"Accepted case fact, no external corroboration ...",supplied_BGCC_FANCODE_R3
7,season_price,"Typically INR399, declining toward season end","Use remaining-season coverage, not rolling ann...",supplied_BGCC_FANCODE_R3


,reading,ordinary_index,spike_index,uplift_pct
0,Stated uplift,100,150,50.000
1,Illustrative bars,100,500,400.000


The bar uplift is 8 times the stated uplift, but the bars are explicitly illustrative. Neither is a measured acquisition multiplier.


## 3. External evidence stays in its own ledger
Third-party estimates differ in population and period. No addition of subscribers, subscriptions, households and fans. Vendor rates can inform costs but cannot measure FanCode conversion.

In [4]:
external=pd.DataFrame(list(METRICS.values()))
display(table(external,'00_external_evidence_register')[['metric_id','value','unit','evidence_type','limitations']])
requirements=[('Q1','Portfolio segmentation: F1, football, MotoGP','10','02,03,04,05,06','2'),('Q2','Passes, player/event bundles, rental, dynamic pricing, competitors','03,07','01,06','4,5'),('Q3','Retention and sustained-growth KPIs','09,10','01,02,04','6,8'),('Q4','Marketing percentages and estimated channel costs/CAC','08','07,09','7,8')]
req=pd.DataFrame(requirements,columns=['question','requirement','primary_notebooks','supporting_notebooks','planned_slides']);display(table(req,'00_requirement_map'))
check('00_evidence',{'locked_raw_pass':json.loads(v.stdout)['status']=='PASS','all_source_ids_reviewed':len(quality)==len(SOURCES),'case_inputs_unique':b.input_id.is_unique})

,metric_id,value,unit,evidence_type,limitations
0,ibm_india_tennis,37,percent,survey_estimate,"Marginal share, not overlap, payers or nationa..."
1,ibm_india_soccer,60,percent,survey_estimate,"Marginal share, not overlap, payers or nationa..."
2,ibm_india_f1,22,percent,survey_estimate,"Marginal share, not overlap, payers or nationa..."
3,ibm_india_cricket,91,percent,survey_estimate,"Marginal share, not overlap, payers or nationa..."
4,yougov_tennis_2024,23,percent,survey_estimate,"Different population from IBM, no averaging"
...,...,...,...,...,...
30,isl_online_2025_26,5.790,million reported online viewers,secondary_reported_audience,Underlying AIFF document and audience definiti...
31,isl_linear_2025_26,9.390,million reported linear TV viewers,secondary_reported_audience,Do not add to online count. Overlap unknown. N...
32,vi_postpaid_pro,199,INR per month listed,partner_public_price,Auto-renew stated. Prepaid Pro differs at INR2...
33,vi_postpaid_plus,248,INR per month listed,partner_public_price,FanCode catalog inclusion does not prove ATP a...


,question,requirement,primary_notebooks,supporting_notebooks,planned_slides
0,Q1,"Portfolio segmentation: F1, football, MotoGP",10,"02,03,04,05,06",2
1,Q2,"Passes, player/event bundles, rental, dynamic ...","03,07","01,06","4,5"
2,Q3,Retention and sustained-growth KPIs,"09,10","01,02,04","6,8"
3,Q4,Marketing percentages and estimated channel co...,08,"07,09","7,8"


,check,passed
0,locked_raw_pass,True
1,all_source_ids_reviewed,True
2,case_inputs_unique,True


## Decision passed forward
Use audience state × occasion × entitlement as the analytical unit. Calendar, pricing and cost evidence can support decisions. Public data cannot establish segment sizes, willingness to pay, internal rights profitability or causal acquisition lift. These become explicit scenarios and pilot gates, not fabricated datasets.